# TEEG Memory Demo — TOON-Encoded Evolving Graph

This notebook gives an interactive, visual walkthrough of the **TEEG** memory
system built into OpenMemoryLab.

**What you'll see:**

1. [Ingest facts](#1.-Ingest-facts) — `MemoryEvolver` distils raw text into `AtomicNote`s and resolves contradictions at write time  
2. [TOON format](#2.-TOON-format) — inspect the compact `key: value` serialisation (~40 % fewer tokens than JSON)  
3. [Graph visualisation](#3.-Graph-visualisation) — render the `TEEGStore` NetworkX DiGraph with `matplotlib`  
4. [Scout retrieval](#4.-Scout-retrieval) — `ScoutRetriever` BFS traversal from seed notes  
5. [Full pipeline query](#5.-Full-pipeline-query) — `TEEGPipeline.query()` end-to-end with TOON context block  
6. [REST API](#6.-REST-API) — call the same logic via the FastAPI server (`/teeg/ingest`, `/teeg/query`)

---

> **Setup** — run from the project root so imports resolve correctly:
> ```
> pip install -e ".[dev]"
> jupyter lab notebooks/teeg_demo.ipynb
> ```

In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on sys.path when running from notebooks/
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")

## 1. Ingest facts

We build a small `TEEGStore` + `MemoryEvolver` and ingest a handful of facts
from Mary Shelley's *Frankenstein*.  Note that facts 3 and 4 **contradict** each
other — the evolver will detect this and archive one of them.

In [ ]:
import tempfile
from oml.memory.atomic_note import AtomicNote
from oml.memory.evolver import MemoryEvolver
from oml.storage.teeg_store import TEEGStore

# Use a temporary directory so the notebook doesn't pollute your working tree
_tmpdir = tempfile.mkdtemp(prefix="teeg_nb_")
store = TEEGStore(artifacts_dir=_tmpdir)
evolver = MemoryEvolver(store, model_name="mock")  # swap to "ollama:qwen3:4b" for real LLM

FACTS = [
    ("Victor Frankenstein assembled the creature from corpse parts over two years.",
     ["frankenstein", "creature", "assembled", "corpse"], ["creation"]),
    ("The creature was brought to life on a stormy November night.",
     ["creature", "life", "night", "stormy"], ["creation"]),
    # Contradicting pair: Victor's reaction immediately after the creature awoke
    ("Victor fled his laboratory immediately after the creature first opened its eyes.",
     ["victor", "fled", "laboratory", "creature"], ["reaction"]),
    ("Victor calmly observed the creature for several minutes before leaving.",
     ["victor", "observed", "creature", "calm"], ["reaction"]),  # ← contradicts previous
    ("The creature learned language by secretly observing the De Lacey family.",
     ["creature", "language", "delacey", "observe"], ["education"]),
    ("Walton encountered Victor adrift on the Arctic ice, near death.",
     ["walton", "victor", "arctic", "adrift"], ["arctic"]),
]

notes = []
for text, keywords, tags in FACTS:
    note = AtomicNote(
        content=text,
        context="Frankenstein demo",
        keywords=keywords,
        tags=tags,
        confidence=0.9,
    )
    evolver.evolve(note)
    notes.append(note)
    status = "✓ active" if note.active else "✗ archived"
    print(f"{status}  {note.note_id[:8]}…  {text[:65]}")

print(f"\nStore: {store.stats()}")

## 2. TOON format

**TOON** (Token-Optimised Object Notation) is a custom serialisation format that
stores `AtomicNote`s as `key: value` pairs — one per line — with pipe-separated
lists.  It uses ~40 % fewer tokens than equivalent JSON when passed to an LLM.

In [ ]:
import json
from oml.memory.toon import encode, decode

example_note = notes[0]  # first AtomicNote

toon_str = encode(example_note)
print("─── TOON encoding ───")
print(toon_str)

# Compare token count (rough word-level estimate)
json_str = json.dumps(example_note.__dict__, default=list, indent=2)
toon_words = len(toon_str.split())
json_words = len(json_str.split())
savings = (1 - toon_words / json_words) * 100

print(f"\n─── Size comparison ───")
print(f"TOON  {len(toon_str):>5} chars  {toon_words:>4} words")
print(f"JSON  {len(json_str):>5} chars  {json_words:>4} words")
print(f"Savings: ~{savings:.0f}% fewer words → fewer tokens in LLM prompt")

# Round-trip verification
restored = decode(toon_str)
assert restored.content == example_note.content, "Round-trip content mismatch!"
print("\nRound-trip decode: OK ✓")

In [ ]:
print("─── Equivalent JSON (for comparison) ───")
print(json_str[:600], "...")

## 3. Graph visualisation

The `TEEGStore` maintains a **NetworkX DiGraph** where each node is an
`AtomicNote` and each directed edge carries a relation label
(`CONTRADICTS`, `EXTENDS`, `SUPPORTS`, `UNRELATED`) and a weight in \[0, 1\].

In [ ]:
try:
    import matplotlib.pyplot as plt
    import networkx as nx
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print("matplotlib not installed — skipping graph plot.")
    print("Install with: pip install matplotlib")

if HAS_MATPLOTLIB:
    G = store.graph  # NetworkX DiGraph

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.set_title("TEEG Memory Graph — AtomicNotes & Relations", fontsize=14, pad=15)

    # Layout
    pos = nx.spring_layout(G, seed=42, k=2.5)

    # Node colours: green = active, grey = archived
    node_colours = []
    node_labels = {}
    for node_id in G.nodes():
        note_obj = store.get(node_id)
        if note_obj:
            colour = "#4caf50" if note_obj.active else "#9e9e9e"
            label = note_obj.content[:30] + "…"
        else:
            colour = "#2196f3"
            label = node_id[:12]
        node_colours.append(colour)
        node_labels[node_id] = label

    # Edge colours by relation
    relation_colour = {
        "CONTRADICTS": "#f44336",  # red
        "EXTENDS":     "#2196f3",  # blue
        "SUPPORTS":    "#4caf50",  # green
        "UNRELATED":   "#9e9e9e",  # grey
    }
    edge_colours = [
        relation_colour.get(G.edges[u, v].get("relation", "UNRELATED"), "#9e9e9e")
        for u, v in G.edges()
    ]
    edge_labels = {
        (u, v): G.edges[u, v].get("relation", "")
        for u, v in G.edges()
    }

    nx.draw_networkx_nodes(G, pos, node_color=node_colours, node_size=1200, ax=ax)
    nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=7, ax=ax)
    nx.draw_networkx_edges(
        G, pos, edge_color=edge_colours, arrows=True,
        arrowstyle="->", arrowsize=15, width=1.8,
        connectionstyle="arc3,rad=0.1", ax=ax
    )
    nx.draw_networkx_edge_labels(
        G, pos, edge_labels=edge_labels, font_size=7,
        label_pos=0.35, ax=ax
    )

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="#4caf50", label="Active note"),
        Patch(facecolor="#9e9e9e", label="Archived note"),
        Patch(facecolor="#f44336", label="CONTRADICTS"),
        Patch(facecolor="#2196f3", label="EXTENDS"),
        Patch(facecolor="#4caf50", label="SUPPORTS"),
    ]
    ax.legend(handles=legend_elements, loc="upper left", fontsize=9)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f"\nGraph summary: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## 4. Scout retrieval

`ScoutRetriever` performs relation-first **BFS** from keyword-matched seed notes.
Each result carries a `(note, score, hops)` tuple so you can see exactly which
notes were retrieved directly versus discovered through graph traversal.

In [ ]:
from oml.retrieval.scout import ScoutRetriever

scout = ScoutRetriever(store, seed_k=2, max_hops=2)

queries = [
    "Who created the creature and how?",
    "What did Victor do right after the creature awoke?",
    "How did the creature learn to speak?",
]

for query in queries:
    print(f"Q: {query!r}")
    results = scout.search(query, top_k=4)
    if not results:
        print("  (no results)")
    for note, score, hops in results:
        label = "seed" if hops == 0 else f"hop-{hops}"
        active = "active" if note.active else "archived"
        print(f"  [{label:<6}  score={score:.3f}  {active}]  {note.content[:72]}")
    print()

In [ ]:
# ScoutRetriever.explain() returns a human-readable traversal summary
print("─── Scout explain ───")
print(scout.explain(queries[1], top_k=4))

## 5. Full pipeline query

`TEEGPipeline` wraps the entire flow — note distillation, evolution, Scout
retrieval, TOON context assembly, and LLM generation — into two methods:
`ingest(raw_text)` and `query(question)`.

In [ ]:
from oml.memory.teeg_pipeline import TEEGPipeline

# Create a fresh pipeline pointing at our temp store
pipeline = TEEGPipeline(model="mock", artifacts_dir=_tmpdir)

question = "Who built the monster and what were the circumstances?"
print(f"Question: {question!r}\n")

answer, context_block = pipeline.query(question, top_k=5, return_context=True)

print("─── LLM Answer ───")
print(answer)

print("\n─── TOON Context Block (what the LLM sees) ───")
print(context_block)

In [ ]:
# Ingest a new fact via the high-level pipeline
new_fact = "Elizabeth Lavenza was Victor's adopted sister and fiancée, murdered by the creature."
print(f"Ingesting: {new_fact!r}\n")

new_note = pipeline.ingest(new_fact, context_hint="Frankenstein, Chapter 23")
pipeline.save()

print(f"note_id  : {new_note.note_id}")
print(f"content  : {new_note.content}")
print(f"keywords : {list(new_note.keywords)[:6]}")
print(f"active   : {new_note.active}")

stats = pipeline.stats()
print(f"\nStore stats after new ingest: {stats}")

## 6. REST API

The exact same TEEG pipeline is exposed via the FastAPI server.  Start it with
`oml api` (or `uvicorn oml.api.server:app`) and then call it with `httpx`.

> **Skip this section** if the server is not running — the cell below handles
> the `ConnectError` gracefully.

In [ ]:
import os

API_BASE = os.getenv("OML_API_BASE", "http://localhost:8000")

try:
    import httpx
    HAS_HTTPX = True
except ImportError:
    HAS_HTTPX = False
    print("httpx not installed — install with:  pip install httpx")

if HAS_HTTPX:
    try:
        with httpx.Client(timeout=10.0) as client:
            # ── /health ──────────────────────────────────────────────────
            r = client.get(f"{API_BASE}/health")
            health = r.json()
            print(f"Server: {API_BASE}")
            print(f"Status : {health['status']}   version: {health['version']}")
            print(f"Storage: {health['storage']}  teeg_ready: {health['teeg_ready']}\n")

            # ── POST /teeg/ingest ─────────────────────────────────────────
            payload = {
                "text": "Walton encountered Victor Frankenstein adrift on the Arctic ice.",
                "context": "Frankenstein, Walton's letters",
            }
            r = client.post(f"{API_BASE}/teeg/ingest", json=payload)
            data = r.json()
            print("POST /teeg/ingest →")
            print(f"  note_id : {data.get('note_id')}")
            print(f"  content : {data.get('content', '')[:70]}")
            print(f"  {data.get('message', '')}\n")

            # ── POST /teeg/query ──────────────────────────────────────────
            payload = {"question": "What happened to Victor at the end of the story?"}
            r = client.post(f"{API_BASE}/teeg/query", json=payload)
            data = r.json()
            print("POST /teeg/query →")
            print(f"  Answer: {data.get('answer', '')[:200]}")
            print(f"  Notes used: {len(data.get('notes_used', []))}")
            print(f"  Latency: {data.get('latency_ms')} ms")

    except httpx.ConnectError:
        print(f"Server not reachable at {API_BASE}")
        print("Start with:  oml api")

---

## Summary

| Component | Role |
|-----------|------|
| `AtomicNote` | Zettelkasten-style memory unit — one fact, full provenance |
| `TOON encoder` | Compact `key: value` serialisation — ~40% fewer tokens than JSON |
| `MemoryEvolver` | LLM-judge resolves contradictions at write time |
| `TEEGStore` | JSON-Lines persistence + NetworkX DiGraph |
| `ScoutRetriever` | Relation-first BFS traversal for retrieval |
| `TEEGPipeline` | End-to-end façade: `ingest()` + `query()` |
| `oml/api/server.py` | FastAPI endpoints: `/teeg/ingest`, `/teeg/query` |

**Next steps:**

- Swap `model="mock"` for `model="ollama:qwen3:4b"` to use a real LLM  
- Run `oml teeg-ingest --file your_doc.txt` to ingest a whole file  
- Run `scripts/teeg.py` for the self-contained command-line demo  
- Open `http://localhost:8000/docs` for the interactive Swagger UI